# Mage iris_classifier: run, tests as a gate, evaluation branch
Run from the repository root. Pipeline: `load_iris -> validate_iris -> prepare_data -> train_model -> [save_results, evaluate_model]`.

In [1]:
import os
if os.path.basename(os.getcwd()) == 'iris_project':
    os.chdir('..')      # run everything from the repository root
print(os.getcwd())

/home/prince_deepak_siddharth/Documents/sem 7/freshcart-pipelines


## Pipeline definition (metadata.yaml)

In [2]:
!cat iris_project/pipelines/iris_classifier/metadata.yaml

name: iris_classifier
uuid: iris_classifier
type: python
blocks:
- uuid: load_iris
  name: load_iris
  type: data_loader
  language: python
  upstream_blocks: []
  downstream_blocks:
  - validate_iris
- uuid: validate_iris
  name: validate_iris
  type: transformer
  language: python
  upstream_blocks:
  - load_iris
  downstream_blocks:
  - prepare_data
- uuid: prepare_data
  name: prepare_data
  type: transformer
  language: python
  upstream_blocks:
  - validate_iris
  downstream_blocks:
  - train_model
- uuid: train_model
  name: train_model
  type: transformer
  language: python
  upstream_blocks:
  - prepare_data
  downstream_blocks:
  - save_results
  - evaluate_model
- uuid: save_results
  name: save_results
  type: data_exporter
  language: python
  upstream_blocks:
  - train_model
  downstream_blocks: []
- uuid: evaluate_model
  name: evaluate_model
  type: transformer
  language: python
  upstream_blocks:
  - train_model
  downstream_blocks: []


## C1. Run with tests

In [3]:
!rm -f iris_predictions.csv
!mage run iris_project iris_classifier --test 2>&1 | grep -vE 'traitlets|DBT|INFO:|FutureWarning'

--------------------------------------------------------------
1/1 tests passed.
validate_iris: 150 rows, 3 classes, 0 missing values
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
max_depth=3  test accuracy=0.9667
--------------------------------------------------------------
1/1 tests passed.
Confusion matrix (rows = true, columns = predicted):
                 pred_setosa  pred_versicolor  pred_virginica
true_setosa               10                0               0
true_versicolor            0                9               1
true_virginica             0                0              10

Metrics:
     class  precision  recall    f1  support
    setosa      1.000     1.0 1.000       10
versicolor      1.000     0.9 0.947       10
 virginica      0.909     1.0 0.952       10
  accuracy        NaN     NaN 0.967       30
saved 30 predictions -> iris_predictions.csv
--------

## C2. max_depth = 1..5 (CSV deleted before each run). depth 1 fails the accuracy test, so save_results never runs

In [4]:
import os, subprocess
for d in range(1, 6):
    if os.path.exists('iris_predictions.csv'): os.remove('iris_predictions.csv')
    r = subprocess.run(['mage','run','iris_project','iris_classifier','--runtime-vars',f'{{"max_depth": {d}}}','--test'], capture_output=True, text=True)
    acc = [l for l in (r.stdout+r.stderr).splitlines() if 'test accuracy' in l]
    ok = 'Pipeline run completed' in (r.stdout+r.stderr)
    print(f'max_depth={d}  {acc[0] if acc else ""}  pipeline completed={ok}  iris_predictions.csv created={os.path.exists("iris_predictions.csv")}')

max_depth=1  max_depth=1  test accuracy=0.6667  pipeline completed=False  iris_predictions.csv created=False


max_depth=2  max_depth=2  test accuracy=0.9333  pipeline completed=True  iris_predictions.csv created=True


max_depth=3  max_depth=3  test accuracy=0.9667  pipeline completed=True  iris_predictions.csv created=True


max_depth=4  max_depth=4  test accuracy=0.9333  pipeline completed=True  iris_predictions.csv created=True


max_depth=5  max_depth=5  test accuracy=0.9333  pipeline completed=True  iris_predictions.csv created=True
